# 05 - Imputación de valores faltantes

**Concepto:** qué hacer con los `NaN`, por qué la **mediana** y no la media, y por qué el imputador también se ajusta sólo con train.

Corresponde al **paso 3 de `preprocess_data()`**.

---

## Por qué no podemos ignorarlos

La mayoría de los estimadores de `scikit-learn` levantan un error si encuentran `NaN`:

```
ValueError: Input contains NaN, infinity or a value too large for dtype('float64').
```

Y en este dataset **más de la mitad** de `EXT_SOURCE_1` está vacía.

## Las opciones

| Estrategia | Cuándo conviene | Costo |
|---|---|---|
| **Eliminar filas** (`dropna`) | Muy pocos nulos, distribuidos al azar | Perdés datos; acá quedarías casi sin dataset |
| **Eliminar columnas** | La columna tiene >90% de nulos y poca señal | Perdés información |
| **Imputar con media** | Distribución simétrica, sin outliers | La media se corre con outliers |
| **Imputar con mediana** ← *el proyecto* | Distribuciones sesgadas (lo habitual en finanzas) | Reduce la varianza real |
| **Imputar + flag de faltante** | Cuando "que falte" es informativo | Duplica columnas |
| **Modelos que aceptan NaN** | LightGBM, XGBoost, HistGradientBoosting | Cambia de modelo |

In [ ]:
import sys
from pathlib import Path

# Permite importar `helpers.py` sin importar desde donde se abra el notebook.
for _p in [Path.cwd(), Path.cwd() / "material_extra", Path.cwd().parent]:
    if (_p / "helpers.py").exists():
        sys.path.insert(0, str(_p))
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import helpers

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
print("Dataset original disponible:", helpers.dataset_disponible())

In [ ]:
from sklearn.model_selection import train_test_split

df = helpers.cargar_muestra(n=20_000)

# Corregimos el valor anomalo visto en el notebook 02
df.loc[df["DAYS_EMPLOYED"] == 365_243, "DAYS_EMPLOYED"] = np.nan

X = df.drop(columns=["TARGET"])
y = df["TARGET"]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"train={X_train.shape}  val={X_val.shape}")

## Cuánto perderíamos con `dropna()`

In [ ]:
print(f"Filas originales:        {len(X_train):,}")
print(f"Filas sin ningún NaN:    {len(X_train.dropna()):,}")
print(f"Se perdería el {(1 - len(X_train.dropna()) / len(X_train)):.1%} del dataset")

Descartado. Hay que imputar.

## Media vs mediana: por qué el proyecto usa mediana

La media es sensible a valores extremos; la mediana no. En datos financieros (ingresos, montos de crédito) **siempre hay una cola larga a la derecha**.

In [ ]:
col = "AMT_INCOME_TOTAL"
serie = X_train[col]

print(f"{col}:")
print(f"  media    : {serie.mean():>15,.0f}")
print(f"  mediana  : {serie.median():>15,.0f}")
print(f"  máximo   : {serie.max():>15,.0f}")
print(f"\nLa media está {serie.mean() / serie.median():.2f}x por encima de la mediana:")
print("señal inequívoca de distribución sesgada.")

fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(serie[serie < serie.quantile(0.99)], bins=60, ax=ax, color="#4c72b0")
ax.axvline(serie.mean(), color="#d1495b", ls="--", lw=2, label=f"media = {serie.mean():,.0f}")
ax.axvline(serie.median(), color="#4c9f70", ls="--", lw=2, label=f"mediana = {serie.median():,.0f}")
ax.set_title(f"Distribución de {col} (recortado al percentil 99)")
ax.legend()
plt.tight_layout()
plt.show()

### Demostración del efecto de un solo outlier

In [ ]:
base = np.array([30_000, 35_000, 40_000, 42_000, 45_000], dtype=float)
con_outlier = np.append(base, 10_000_000)

print(f"Sin outlier -> media: {base.mean():>12,.0f}   mediana: {np.median(base):>12,.0f}")
print(f"Con outlier -> media: {con_outlier.mean():>12,.0f}   mediana: {np.median(con_outlier):>12,.0f}")
print("\nUn solo valor extremo movió la media 40x. La mediana casi no se movió.")

## `SimpleImputer` en la práctica

Igual que con el encoding y el escalado: **`fit` en train, `transform` en todo**.

In [ ]:
from sklearn.impute import SimpleImputer

numericas = X_train.select_dtypes(include=["int64", "float64"]).columns

imputer = SimpleImputer(strategy="median")
imputer.fit(X_train[numericas])          # aprende UNA mediana por columna

train_imp = pd.DataFrame(imputer.transform(X_train[numericas]),
                         columns=numericas, index=X_train.index)
val_imp = pd.DataFrame(imputer.transform(X_val[numericas]),
                       columns=numericas, index=X_val.index)

print("Medianas aprendidas (statistics_):")
for c, v in list(zip(numericas, imputer.statistics_))[:8]:
    print(f"  {c:22s} {v:>14,.2f}")

print(f"\nNaN en train antes: {X_train[numericas].isnull().sum().sum():,}")
print(f"NaN en train después: {train_imp.isnull().sum().sum()}")
print(f"NaN en val después:   {val_imp.isnull().sum().sum()}")

## El efecto secundario: imputar deforma la distribución

Imputar con una constante crea un **pico artificial** en ese valor. No es gratis.

In [ ]:
col = "EXT_SOURCE_1"
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

sns.histplot(X_train[col].dropna(), bins=40, ax=axes[0], color="#4c72b0")
axes[0].set_title(f"{col}: sólo valores observados")

sns.histplot(train_imp[col], bins=40, ax=axes[1], color="#d1495b")
axes[1].set_title(f"{col}: después de imputar con la mediana")

plt.tight_layout()
plt.show()

print(f"Antes  -> media {X_train[col].mean():.4f}   desvío {X_train[col].std():.4f}")
print(f"Después-> media {train_imp[col].mean():.4f}   desvío {train_imp[col].std():.4f}")
print("\nEl desvío BAJA: le estamos diciendo al modelo que hay menos variabilidad de la real.")

## La mejora fácil: guardar la información del faltante

Antes de imputar, guardamos una columna binaria que indica que el valor faltaba. Así el modelo puede aprender que *"no tener EXT_SOURCE_1"* es en sí mismo una señal (lo verificamos en el notebook 02).

In [ ]:
cols_con_nulos = [c for c in numericas if X_train[c].isnull().any()]

train_flags = X_train[cols_con_nulos].isnull().astype(int)
train_flags.columns = [f"{c}_FALTANTE" for c in cols_con_nulos]

train_mejorado = pd.concat([train_imp, train_flags], axis=1)

print(f"Columnas: {train_imp.shape[1]} -> {train_mejorado.shape[1]}")
print()
print("Tasa de default según cada flag:")
for c in cols_con_nulos:
    tasas = y_train.groupby(X_train[c].isnull()).mean() * 100
    if len(tasas) == 2:
        print(f"  {c:20s} presente={tasas[False]:5.2f}%   ausente={tasas[True]:5.2f}%   "
              f"(dif {tasas[True] - tasas[False]:+.2f} pp)")

Cuando la diferencia es de más de ~1 punto porcentual, esa flag suele aportar. **Es uno de los ejercicios opcionales más rentables del proyecto.**

## Sobre las categóricas

`SimpleImputer(strategy="median")` sólo aplica a numéricas. Para categóricas las opciones son `strategy="most_frequent"` o, mejor, `strategy="constant", fill_value="Desconocido"`, que trata *"no sé"* como una categoría más en lugar de mentir con la moda.

---

## Ejercicios

1. Compará el AUC de una regresión logística con `strategy="median"` vs `strategy="mean"`. ¿Cambia mucho? ¿Por qué (o por qué no)?
2. Imputá las categóricas con `SimpleImputer(strategy="constant", fill_value="Desconocido")` y verificá qué pasa después con el `OneHotEncoder`.
3. Probá `IterativeImputer` (imputación multivariada: predice cada columna faltante a partir de las demás). ¿Mejora el AUC? ¿Vale el costo computacional?
4. **Para discutir:** si imputás *antes* de partir train/val, ¿qué tipo de leakage estás cometiendo? Volvé al notebook 03.

In [ ]:
# Tu turno
